# IIC3745 — Actividad 2: Mutation Testing con LLMs

La Actividad 1 midió **cobertura**. Esta mide algo distinto: si los tests además
**detectan fallas**.

Trabajas sobre el mismo `ReservationService`, con una batería de 47 tests que ya
alcanza **100% de statement y branch coverage**. La pregunta es cuántos mutantes
mata esa batería, y si los mutantes que propone un LLM son mejores o peores que
los de un generador clásico de operadores.

**Flujo de trabajo:**

1. Verifica que la suite base pasa y cubre el 100% (celda de la sección 0).
2. Ejecuta `mostrar_prompt()` y copia el bloque completo.
3. Pégalo en un **chat nuevo** de GPT. Guarda el JSON en `mutantes/gpt.json` y la
   respuesta completa en `prompts/gpt.md`.
4. Repite con Gemini → `mutantes/gemini.json` y `prompts/gemini.md`.
5. Evalúa cada tanda, marca los equivalentes, y compara contra el baseline clásico.

**Categorías de mutante:**

| Estado | Qué significa |
|---|---|
| `killed` | Al menos un test de la batería base falla. El mutante fue detectado. |
| `sobrevive` | Todos los tests pasan. La batería no detecta ese cambio. |
| `invalido` | No se pudo aplicar: la línea no calza, no compila, o no cambia nada. |
| `equivalente` | Sobrevive **y** tú argumentas que es semánticamente idéntico al original. |

El score se calcula como `killed / (total − inválidos − equivalentes)`. Descontar
inválidos y equivalentes es lo que hace que el número signifique algo.

In [2]:
# ============================================================
# CELDA SELLADA - NO MODIFICAR
# ============================================================
import os, sys, hashlib, subprocess

# --- Google Colab -------------------------------------------------------
# Si trabajas en Colab: sube el .zip de la actividad con el panel de archivos
# (icono de carpeta a la izquierda), descomenta las dos lineas siguientes y
# ajusta el nombre del archivo. Ejecuta esta celda una sola vez.
#
# !unzip -q -o Actividad2.zip -d actividad2
# RAIZ = "actividad2"
#
# Si trabajas localmente, deja RAIZ como esta.
RAIZ = "."

if os.path.abspath(RAIZ) != os.getcwd():
    os.chdir(RAIZ)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

try:
    import coverage
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "coverage"], check=True)
    import coverage

HASHES = {
    "src/reservation.py": "d304abfad773ef36",
    "src/models.py": "3d034e6de8e5c775",
    "tests/test_base.py": "0ca6085e7c65bed8",
}

faltantes = [d for d in ("src", "tests", "mutantes", "prompts") if not os.path.isdir(d)]
if faltantes:
    raise SystemExit("Faltan carpetas: %s. Revisa RAIZ." % ", ".join(faltantes))

alterados = [r for r, esp in HASHES.items()
             if hashlib.sha256(open(r, "rb").read()).hexdigest()[:16] != esp]

print("coverage", coverage.__version__, "| python", sys.version.split()[0])
print("directorio:", os.getcwd())
if alterados:
    print("\nADVERTENCIA: estos archivos fueron modificados y no deben serlo:")
    for a in alterados:
        print("   -", a)
    print("Restauralos antes de continuar o tus resultados no seran comparables.")
else:
    print("src/ y tests/ intactos.")

coverage 7.15.4 | python 3.12.4
directorio: c:\Users\vicen\Desktop\sharedrepo\testing\act2
src/ y tests/ intactos.


In [3]:
# ============================================================
# CELDA SELLADA - NO MODIFICAR
# ============================================================
import ast
import importlib
import io
import json
import os
import platform
import re
import signal
import sys
import tokenize
import types
import unittest

import coverage


RUTA_SUT = "src/reservation.py"
MODULO_SUT = "src.reservation"
MODULO_TESTS = "tests.test_base"
MODELOS = ["gpt", "gemini"]
TIMEOUT_SEG = 10

RESULTADOS = {}


# ==========================================================================
# 0. Verificacion previa: la suite base debe pasar y cubrir el 100%
# ==========================================================================

def verificar_base():
    """Sin esto, todo mutante se marca 'killed' por un test que ya fallaba y el
    mutation score no significa nada."""
    cov = coverage.Coverage(branch=True, include=[RUTA_SUT], data_file=None)
    cov.start()
    try:
        res = _correr_suite(fuente=None, failfast=False)
    finally:
        cov.stop()

    tmp = ".cob_tmp.json"
    cov.json_report(outfile=tmp)
    datos = json.load(open(tmp, encoding="utf-8"))
    os.remove(tmp)
    clave = [k for k in datos["files"] if k.replace("\\", "/").endswith("reservation.py")]
    s = datos["files"][clave[0]]["summary"]
    stmt = 100.0 * s["covered_lines"] / s["num_statements"]
    br = 100.0 * s["covered_branches"] / s["num_branches"]

    ok_tests = res.wasSuccessful()
    ok_cov = stmt >= 100 and br >= 100
    print("Suite base: %d tests, %d fallas, %d errores"
          % (res.testsRun, len(res.failures), len(res.errors)))
    print("Cobertura : %.1f%% statement, %.1f%% branch" % (stmt, br))
    if ok_tests and ok_cov:
        print("\n[OK] La suite base pasa y cubre el 100%. Puedes evaluar mutantes.")
    else:
        if not ok_tests:
            print("\n[ERROR] La suite base NO pasa. Todo mutante se marcaria como")
            print("        killed por un test que ya fallaba. Detente aca.")
        if not ok_cov:
            print("\n[ERROR] La suite base no cubre el 100%. Los mutantes en lineas")
            print("        no cubiertas nunca podrian morir.")
    return ok_tests and ok_cov


# ==========================================================================
# 1. Prompt con el codigo numerado
# ==========================================================================

PLANTILLA = """Below is a Python file. Each line is prefixed with its line number.

Generate exactly {n} mutants for this code. A mutant is a small change to a \
SINGLE line that alters the program's behaviour, in the style of mutation testing.

Rules:
- Each mutant modifies exactly one line.
- `precode` must be the ORIGINAL line, copied character by character, including \
its leading indentation, exactly as it appears at that line number.
- `aftercode` is the mutated version of that same line, keeping valid Python \
syntax and the same indentation.
- Do not mutate comments, blank lines, import statements or the class/def \
headers.
- Prefer semantic changes (boundaries, operators, constants, conditions) over \
cosmetic ones.
- Do not repeat the same mutant twice.

Answer with a JSON array and nothing else. No explanation, no markdown fences.

IMPORTANT: the code contains double-quoted Python strings. Inside a JSON string \
every double quote MUST be escaped as \\". Look carefully at the second example \
below: the quotes around "pending" are escaped.

[
  {{"id": 1, "line": 12, "precode": "        if x >= 10:", "aftercode": "        if x > 10:"}},
  {{"id": 2, "line": 34, "precode": "        if s.status != \\"pending\\":", "aftercode": "        if s.status == \\"pending\\":"}},
  ...
]

Here is the code:

{codigo}
"""


def mostrar_prompt(n=20):
    lineas = open(RUTA_SUT, encoding="utf-8").read().splitlines()
    ancho = len(str(len(lineas)))
    numerado = "\n".join("%*d | %s" % (ancho, i, l) for i, l in enumerate(lineas, 1))
    print("=" * 78)
    print("COPIA DESDE LA SIGUIENTE LINEA HASTA EL FINAL, EN UN CHAT NUEVO")
    print("=" * 78)
    print()
    print(PLANTILLA.format(n=n, codigo=numerado))
    print("=" * 78)
    print("FIN. Guarda la respuesta cruda en prompts/<modelo>.md y la lista")
    print("     JSON en mutantes/<modelo>.json")
    print("=" * 78)


# ==========================================================================
# 2. Aplicacion de un mutante
# ==========================================================================

class MutanteInvalido(Exception):
    pass


def _fuente_original():
    return open(RUTA_SUT, encoding="utf-8").read()


_LITERAL_SIMPLE = re.compile(r"'([^'\\\n]*)'")

# artefactos tipicos de copiar y pegar desde una ventana de chat
_ARTEFACTOS = {
    "\u2018": "'", "\u2019": "'",          # comillas simples tipograficas
    "\u201c": '"', "\u201d": '"',          # comillas dobles tipograficas
    "\u00a0": " ", "\u202f": " ",          # espacios no separables
    "\u2013": "-", "\u2014": "-",          # guiones largos
}


def _limpiar(linea):
    for malo, bueno in _ARTEFACTOS.items():
        linea = linea.replace(malo, bueno)
    return linea


def _normalizar(linea):
    """Comillas simples -> dobles en literales sin escapes. En Python 'gold' y
    "gold" son el mismo string, asi que un modelo que reescribe el estilo de
    comillas no esta proponiendo un mutante distinto: solo lo transcribio de
    otra forma."""
    return _LITERAL_SIMPLE.sub(
        lambda m: '"%s"' % m.group(1) if '"' not in m.group(1) else m.group(0),
        _limpiar(linea))


def _tokens(linea):
    """Secuencia de tokens significativos de una linea, con los literales
    comparados por VALOR y no por su escritura. Devuelve None si la linea no
    tokeniza (fragmento incompleto, sintaxis rota)."""
    try:
        flujo = tokenize.generate_tokens(io.StringIO(_limpiar(linea).strip()).readline)
        salida = []
        for tok in flujo:
            if tok.type in (tokenize.NEWLINE, tokenize.NL, tokenize.INDENT,
                            tokenize.DEDENT, tokenize.ENDMARKER, tokenize.COMMENT):
                continue
            if tok.type == tokenize.STRING:
                try:
                    salida.append(("STR", ast.literal_eval(tok.string)))
                    continue
                except (ValueError, SyntaxError):
                    pass
            salida.append((tok.type, tok.string))
        return tuple(salida)
    except (tokenize.TokenError, IndentationError, SyntaxError):
        return None


def _equivalentes(a, b):
    """True si dos lineas son el mismo codigo salvo espacios, estilo de comillas
    y artefactos de copiado."""
    if _normalizar(a).strip() == _normalizar(b).strip():
        return True
    ta, tb = _tokens(a), _tokens(b)
    return ta is not None and ta == tb


def _calzar(real, precode):
    """Devuelve (calza, motivo_de_ajuste). Va de mas a menos estricto; todos los
    ajustes son de transcripcion, no de semantica."""
    if real == precode:
        return True, None
    if _normalizar(real) == _normalizar(precode):
        return True, "comillas"
    if _normalizar(real).strip() == _normalizar(precode).strip():
        return True, "indentacion"
    ta, tb = _tokens(real), _tokens(precode)
    if ta is not None and ta == tb:
        return True, "espaciado"
    return False, None


def aplicar(mutante, fuente=None):
    """Devuelve la fuente con el mutante aplicado. Lanza MutanteInvalido si el
    mutante no calza con el codigo."""
    if fuente is None:
        fuente = _fuente_original()
    lineas = fuente.splitlines()

    for campo in ("line", "precode", "aftercode"):
        if campo not in mutante:
            raise MutanteInvalido("falta el campo '%s'" % campo)

    n = mutante["line"]
    if not isinstance(n, int) or n < 1 or n > len(lineas):
        raise MutanteInvalido("linea %r fuera de rango (1..%d)" % (n, len(lineas)))

    real = lineas[n - 1]
    calza, ajuste = _calzar(real, mutante["precode"])
    if not calza:
        raise MutanteInvalido(
            "precode no calza en la linea %d\n      esperado: %r\n      real    : %r"
            % (n, mutante["precode"], real))

    nueva_linea = _limpiar(mutante["aftercode"])
    if ajuste in ("indentacion", "espaciado"):
        sangria = real[:len(real) - len(real.lstrip())]
        nueva_linea = sangria + nueva_linea.lstrip()
    mutante["_ajuste"] = ajuste

    if _equivalentes(nueva_linea, real):
        raise MutanteInvalido("aftercode identico a precode (no cambia nada)")

    lineas[n - 1] = nueva_linea
    nueva = "\n".join(lineas) + "\n"

    try:
        compile(nueva, RUTA_SUT, "exec")
    except SyntaxError as e:
        raise MutanteInvalido("el codigo mutado no compila: %s" % e)
    return nueva


# ==========================================================================
# 3. Ejecucion de la suite contra una fuente dada
# ==========================================================================

class _Timeout(Exception):
    pass


def _alarma(signum, frame):
    raise _Timeout()


def _correr_suite(fuente, failfast=True):
    for m in list(sys.modules):
        if m == "src" or m.startswith("src.") or m.startswith("tests."):
            del sys.modules[m]

    if fuente is not None:
        mod = types.ModuleType(MODULO_SUT)
        mod.__file__ = os.path.abspath(RUTA_SUT)
        mod.__package__ = "src"
        sys.modules[MODULO_SUT] = mod
        exec(compile(fuente, mod.__file__, "exec"), mod.__dict__)

    mod_tests = importlib.import_module(MODULO_TESTS)
    suite = unittest.defaultTestLoader.loadTestsFromModule(mod_tests)
    runner = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0, failfast=failfast)
    return runner.run(suite)


def _evaluar_uno(mutante):
    """Devuelve (estado, detalle). Estados: killed, sobrevive, invalido."""
    try:
        fuente = aplicar(mutante)
    except MutanteInvalido as e:
        return "invalido", str(e)

    usar_alarma = platform.system() != "Windows" and hasattr(signal, "SIGALRM")
    if usar_alarma:
        anterior = signal.signal(signal.SIGALRM, _alarma)
        signal.alarm(TIMEOUT_SEG)
    try:
        res = _correr_suite(fuente)
    except _Timeout:
        return "killed", "timeout (%ds)" % TIMEOUT_SEG
    except Exception as e:
        # el modulo mutado revienta al importarse: ningun test puede pasar
        return "killed", "error al cargar el modulo: %s" % type(e).__name__
    finally:
        if usar_alarma:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, anterior)

    if res.wasSuccessful():
        return "sobrevive", ""
    culpable = (res.failures + res.errors)[0][0]
    return "killed", str(culpable).split(" ")[0]


# ==========================================================================
# 4. Carga y evaluacion de una tanda de mutantes
# ==========================================================================

_CAMPO = re.compile(
    r'("(?:precode|aftercode|justificacion)"\s*:\s*")(.*?)("\s*(?:,|\}))', re.S)


def _reparar_json(texto):
    """Los modelos suelen devolver el codigo Python sin escapar sus comillas
    dobles internas:

        "precode": "        if reservation.status != "pending":"

    Eso rompe el JSON entero y ningun mutante se puede evaluar. Aca se reescapan
    las comillas del interior de precode/aftercode/justificacion. El terminador
    del campo se reconoce como una comilla seguida de coma o llave, que es lo
    unico que no puede aparecer dentro de una linea de codigo bien formada."""
    def sub(m):
        cuerpo = m.group(2).replace('\\"', '"').replace('"', '\\"')
        return m.group(1) + cuerpo + m.group(3)
    return _CAMPO.sub(sub, texto)


def cargar_mutantes(ruta):
    texto = open(ruta, encoding="utf-8").read().strip()
    if not texto:
        raise ValueError("%s esta vacio" % ruta)
    if texto.startswith("```"):
        texto = "\n".join(l for l in texto.splitlines() if not l.strip().startswith("```"))
    texto = re.sub(r",(\s*[\]\}])", r"\1", texto)      # comas colgantes

    try:
        datos = json.loads(texto)
    except json.JSONDecodeError as original:
        try:
            datos = json.loads(_reparar_json(texto))
        except json.JSONDecodeError:
            raise ValueError(
                "%s no es JSON valido y no se pudo reparar automaticamente.\n"
                "  Error original: %s\n"
                "  Revisa que sea una lista de objetos con los campos line, "
                "precode y aftercode." % (ruta, original))
        print("  [aviso] %s tenia comillas internas sin escapar; se reparo "
              "automaticamente." % ruta)
        print("          El archivo en disco NO se modifico: se conserva la "
              "respuesta cruda del modelo.")

    if not isinstance(datos, list):
        raise ValueError("%s debe contener una lista JSON" % ruta)
    for i, m in enumerate(datos, 1):
        m.setdefault("id", i)
    return datos


def evaluar(nombre, mutantes=None, mostrar=True):
    """nombre: 'gpt', 'gemini' o 'clasico'."""
    if mutantes is None:
        mutantes = cargar_mutantes("mutantes/%s.json" % nombre)

    filas = []
    for m in mutantes:
        estado, detalle = _evaluar_uno(m)
        ajuste = m.pop("_ajuste", None)
        if estado == "sobrevive" and m.get("equivalente"):
            estado = "equivalente"
        filas.append({"id": m.get("id"), "line": m.get("line"), "estado": estado,
                      "ajuste": ajuste,
                      "detalle": detalle, "precode": m.get("precode", ""),
                      "aftercode": m.get("aftercode", ""),
                      "justificacion": m.get("justificacion", "")})

    cuenta = {e: sum(1 for f in filas if f["estado"] == e)
              for e in ("killed", "sobrevive", "equivalente", "invalido")}
    total = len(filas)
    denom = total - cuenta["invalido"] - cuenta["equivalente"]
    score = 100.0 * cuenta["killed"] / denom if denom else 0.0

    r = {"nombre": nombre, "total": total, "validos": total - cuenta["invalido"],
         "invalidos": cuenta["invalido"], "equivalentes": cuenta["equivalente"],
         "killed": cuenta["killed"], "sobreviven": cuenta["sobrevive"],
         "denominador": denom, "score": round(score, 1), "filas": filas}
    RESULTADOS[nombre] = r

    if mostrar:
        _imprimir(r)
    return r


def _imprimir(r):
    print("Mutantes de: %s" % r["nombre"].upper())
    print("-" * 64)
    print("  Total propuestos ................. %d" % r["total"])
    print("  Invalidos (no aplican) ........... %d" % r["invalidos"])
    print("  Validos .......................... %d" % r["validos"])
    print("  Equivalentes (marcados) .......... %d" % r["equivalentes"])
    print("  Killed ........................... %d" % r["killed"])
    print("  Sobreviven ....................... %d" % r["sobreviven"])
    print("  Mutation score ajustado .......... %.1f%%  (%d / %d)"
          % (r["score"], r["killed"], r["denominador"]))

    ajustados = [f for f in r["filas"] if f.get("ajuste")]
    if ajustados:
        porq = {}
        for f in ajustados:
            porq[f["ajuste"]] = porq.get(f["ajuste"], 0) + 1
        print("\n  [aviso] %d mutante(s) calzaron tras normalizar %s. Son diferencias"
              % (len(ajustados), " y ".join(sorted(porq))))
        print("          de transcripcion, no de semantica: 'gold' y \"gold\" son el mismo")
        print("          string. Se evaluaron normalmente; mencionalo en el reporte.")

    invalidos = [f for f in r["filas"] if f["estado"] == "invalido"]
    if invalidos:
        print("\n  Mutantes invalidos:")
        for f in invalidos:
            print("    #%-3s linea %-4s %s" % (f["id"], f["line"], f["detalle"]))

    vivos = [f for f in r["filas"] if f["estado"] == "sobrevive"]
    if vivos:
        print("\n  SOBREVIVIENTES (%d) - analizalos en el reporte:" % len(vivos))
        for f in vivos:
            print("    #%-3s linea %-4s" % (f["id"], f["line"]))
            print("         antes : %s" % f["precode"].strip())
            print("         despues: %s" % f["aftercode"].strip())

    equis = [f for f in r["filas"] if f["estado"] == "equivalente"]
    if equis:
        print("\n  Marcados como equivalentes (%d):" % len(equis))
        for f in equis:
            print("    #%-3s linea %-4s %s" % (f["id"], f["line"],
                                               f["justificacion"][:60] or "SIN JUSTIFICACION"))


# ==========================================================================
# 5. Baseline: mutantes clasicos generados con operadores estandar
# ==========================================================================

SUSTITUCIONES = {
    "<": ["<=", ">"], "<=": ["<", ">="], ">": [">=", "<"], ">=": [">", "<="],
    "==": ["!="], "!=": ["=="],
    "+": ["-"], "-": ["+"], "*": ["/"], "/": ["*"], "//": ["/"],
}
PALABRAS = {"and": ["or"], "or": ["and"], "in": ["not in"], "is": ["is not"]}


def generar_clasicos():
    """Mutantes de operadores estandar (ROR, AOR, COR, constantes), en el mismo
    formato precode/aftercode. Usa tokenize, asi que nunca toca strings ni
    comentarios."""
    fuente = _fuente_original()
    lineas = fuente.splitlines()
    saltar = set()
    for nodo in ast.walk(ast.parse(fuente)):
        if isinstance(nodo, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.ClassDef)):
            saltar.add(nodo.lineno)

    mutantes, vistos = [], set()
    with open(RUTA_SUT, "rb") as fh:
        toks = list(tokenize.tokenize(fh.readline))
    utiles = [t for t in toks
              if t.type not in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE,
                                tokenize.INDENT, tokenize.DEDENT, tokenize.ENCODING)]
    for idx, tok in enumerate(utiles):
        fila, col = tok.start
        if tok.start[0] != tok.end[0] or fila in saltar:
            continue
        anterior = utiles[idx - 1].string if idx else ""
        siguiente = utiles[idx + 1].string if idx + 1 < len(utiles) else ""
        opciones = []
        if tok.type == tokenize.OP:
            opciones = SUSTITUCIONES.get(tok.string, [])
        elif tok.type == tokenize.NAME:
            if tok.string == "in" and anterior == "not":
                continue                      # evita 'not not in'
            if tok.string == "is" and siguiente == "not":
                continue                      # evita 'is not not'
            opciones = PALABRAS.get(tok.string, [])
        elif tok.type == tokenize.NUMBER:
            try:
                val = float(tok.string)
                opciones = [x for x in ("0", repr(val + 1)) if x != tok.string]
            except ValueError:
                opciones = []
        for nuevo in opciones:
            original = lineas[fila - 1]
            mutada = original[:col] + nuevo + original[col + len(tok.string):]
            clave = (fila, mutada)
            if mutada == original or clave in vistos:
                continue
            vistos.add(clave)
            mutantes.append({"id": len(mutantes) + 1, "line": fila,
                             "precode": original, "aftercode": mutada})
    return mutantes


# ==========================================================================
# 6. Consolidacion
# ==========================================================================

def consolidar():
    faltan = [m for m in MODELOS + ["clasico"] if m not in RESULTADOS]
    if faltan:
        print("Faltan por evaluar: %s\n" % ", ".join(faltan))

    print("## Tabla 1 - Mutantes por fuente\n")
    print("| Fuente | Total | Invalidos | Validos | Equivalentes | Killed | "
          "Sobreviven | Score ajustado |")
    print("|---|---|---|---|---|---|---|---|")
    for n in MODELOS + ["clasico"]:
        r = RESULTADOS.get(n)
        if r is None:
            print("| %s | - | - | - | - | - | - | - |" % n)
            continue
        print("| %s | %d | %d | %d | %d | %d | %d | %.1f%% |" % (
            n, r["total"], r["invalidos"], r["validos"], r["equivalentes"],
            r["killed"], r["sobreviven"], r["score"]))

    print("\n" + "-" * 70)
    avisos = []
    for n in MODELOS:
        r = RESULTADOS.get(n)
        if not r:
            continue
        if r["invalidos"] > r["total"] * 0.3:
            avisos.append("%s: mas de un tercio de los mutantes son invalidos" % n)
        if r["score"] >= 100 and r["total"] > 0:
            avisos.append("%s: score 100%%. No falta nada que agregar; discute en el "
                          "reporte si eso indica mutantes triviales." % n)
        sin_just = [f for f in r["filas"]
                    if f["estado"] == "equivalente" and not f["justificacion"].strip()]
        if sin_just:
            avisos.append("%s: %d equivalente(s) sin justificacion"
                          % (n, len(sin_just)))
    if avisos:
        print("REVISAR:")
        for a in avisos:
            print("  - " + a)
    else:
        print("Sin observaciones automaticas.")
    print("Copia la tabla a REPORTE.md.")

## 0. Verificación previa

**Esta celda no es opcional.** Si la batería base no pasara, todos los mutantes
quedarían marcados como `killed` por un test que ya venía fallando, y el mutation
score sería 100% sin significar nada. Si no cubriera el 100%, los mutantes en
líneas no cubiertas nunca podrían morir.

Ejecútala antes de cualquier otra cosa.

In [4]:
verificar_base()

Suite base: 47 tests, 0 fallas, 0 errores
Cobertura : 100.0% statement, 100.0% branch

[OK] La suite base pasa y cubre el 100%. Puedes evaluar mutantes.


True

## 1. Generar el prompt

La celda siguiente imprime el código **con números de línea explícitos**. Eso
importa: sin numeración, el modelo cuenta mal y la mayoría de sus mutantes salen
con la línea corrida en uno, lo que los vuelve inválidos.

Pide 20 mutantes por modelo. El formato de respuesta es un array JSON con
`line`, `precode` y `aftercode`.

Usa el **mismo prompt** para GPT y para Gemini: lo que se compara es el modelo, no
la instrucción.

In [5]:
mostrar_prompt(20)

COPIA DESDE LA SIGUIENTE LINEA HASTA EL FINAL, EN UN CHAT NUEVO

Below is a Python file. Each line is prefixed with its line number.

Generate exactly 20 mutants for this code. A mutant is a small change to a SINGLE line that alters the program's behaviour, in the style of mutation testing.

Rules:
- Each mutant modifies exactly one line.
- `precode` must be the ORIGINAL line, copied character by character, including its leading indentation, exactly as it appears at that line number.
- `aftercode` is the mutated version of that same line, keeping valid Python syntax and the same indentation.
- Do not mutate comments, blank lines, import statements or the class/def headers.
- Prefer semantic changes (boundaries, operators, constants, conditions) over cosmetic ones.
- Do not repeat the same mutant twice.

Answer with a JSON array and nothing else. No explanation, no markdown fences.

IMPORTANT: the code contains double-quoted Python strings. Inside a JSON string every double quote MUST b

## 2. Evaluar cada tanda

Pega el JSON del modelo en `mutantes/gpt.json` (y `mutantes/gemini.json`), luego
ejecuta la celda correspondiente.

Es **esperable** que algunos mutantes salgan inválidos: el modelo copia mal una
línea, propone un cambio que no compila, o repite el original sin cambiarlo. Eso
no se corrige y no se descarta en silencio — se reporta. La tasa de mutantes
inválidos es uno de los resultados que compara a los dos modelos.

No edites los mutantes para "arreglarlos". Si lo haces, dejas de medir al modelo.

**Si el modelo devolvió comillas dobles sin escapar** (por ejemplo `"precode": "        if s.status != "pending":"`), el notebook lo detecta y repara el JSON al cargarlo, avisando por pantalla. No repitas el prompt ni edites el archivo: la respuesta cruda del modelo se conserva tal cual en `mutantes/`, que es lo que se evalúa como evidencia.

**Si el modelo transcribió el código con otro estilo de comillas** (`'gold'` en vez de `"gold"`) **perdió la indentación, cambió los espacios o trajo comillas tipográficas del chat**, el notebook lo normaliza y evalúa el mutante igual: en Python son el mismo string, así que no es un mutante distinto sino la misma propuesta transcrita de otra forma. Te avisa cuántos ajustó; mencionálo en el reporte. Lo que sí sigue contando como inválido es la línea equivocada, el código que no compila y el mutante que no cambia nada.


In [7]:
evaluar("gpt")

Mutantes de: GPT
----------------------------------------------------------------
  Total propuestos ................. 20
  Invalidos (no aplican) ........... 0
  Validos .......................... 20
  Equivalentes (marcados) .......... 0
  Killed ........................... 10
  Sobreviven ....................... 10
  Mutation score ajustado .......... 50.0%  (10 / 20)

  [aviso] 20 mutante(s) calzaron tras normalizar indentacion. Son diferencias
          de transcripcion, no de semantica: 'gold' y "gold" son el mismo
          string. Se evaluaron normalmente; mencionalo en el reporte.

  SOBREVIVIENTES (10) - analizalos en el reporte:
    #3   linea 21  
         antes : if not isinstance(nights, int) or nights < 1 or nights > 30:
         despues: if not isinstance(nights, int) or nights <= 1 or nights > 30:
    #4   linea 21  
         antes : if not isinstance(nights, int) or nights < 1 or nights > 30:
         despues: if not isinstance(nights, int) or nights < 1 or nights >= 

{'nombre': 'gpt',
 'total': 20,
 'validos': 20,
 'invalidos': 0,
 'equivalentes': 0,
 'killed': 10,
 'sobreviven': 10,
 'denominador': 20,
 'score': 50.0,
 'filas': [{'id': 1,
   'line': 17,
   'estado': 'killed',
   'ajuste': 'indentacion',
   'detalle': 'test_codigo_duplicado',
   'precode': ' if code in self.reservations:',
   'aftercode': ' if code not in self.reservations:',
   'justificacion': ''},
  {'id': 2,
   'line': 19,
   'estado': 'killed',
   'ajuste': 'indentacion',
   'detalle': 'test_codigo_duplicado',
   'precode': ' if tier not in VALID_TIERS:',
   'aftercode': ' if tier in VALID_TIERS:',
   'justificacion': ''},
  {'id': 3,
   'line': 21,
   'estado': 'sobrevive',
   'ajuste': 'indentacion',
   'detalle': '',
   'precode': ' if not isinstance(nights, int) or nights < 1 or nights > 30:',
   'aftercode': ' if not isinstance(nights, int) or nights <= 1 or nights > 30:',
   'justificacion': ''},
  {'id': 4,
   'line': 21,
   'estado': 'sobrevive',
   'ajuste': 'indentac

In [8]:
evaluar("gemini")

Mutantes de: GEMINI
----------------------------------------------------------------
  Total propuestos ................. 20
  Invalidos (no aplican) ........... 0
  Validos .......................... 20
  Equivalentes (marcados) .......... 0
  Killed ........................... 10
  Sobreviven ....................... 10
  Mutation score ajustado .......... 50.0%  (10 / 20)

  [aviso] 5 mutante(s) calzaron tras normalizar comillas. Son diferencias
          de transcripcion, no de semantica: 'gold' y "gold" son el mismo
          string. Se evaluaron normalmente; mencionalo en el reporte.

  SOBREVIVIENTES (10) - analizalos en el reporte:
    #3   linea 21  
         antes : if not isinstance(nights, int) or nights < 1 or nights > 30:
         despues: if not isinstance(nights, int) or nights <= 1 or nights > 30:
    #5   linea 26  
         antes : if check_in < now:
         despues: if check_in <= now:
    #6   linea 28  
         antes : if (check_in - now).days > 365:
         des

{'nombre': 'gemini',
 'total': 20,
 'validos': 20,
 'invalidos': 0,
 'equivalentes': 0,
 'killed': 10,
 'sobreviven': 10,
 'denominador': 20,
 'score': 50.0,
 'filas': [{'id': 1,
   'line': 17,
   'estado': 'killed',
   'ajuste': None,
   'detalle': 'test_codigo_duplicado',
   'precode': '        if code in self.reservations:',
   'aftercode': '        if code not in self.reservations:',
   'justificacion': ''},
  {'id': 2,
   'line': 19,
   'estado': 'killed',
   'ajuste': None,
   'detalle': 'test_codigo_duplicado',
   'precode': '        if tier not in VALID_TIERS:',
   'aftercode': '        if tier in VALID_TIERS:',
   'justificacion': ''},
  {'id': 3,
   'line': 21,
   'estado': 'sobrevive',
   'ajuste': None,
   'detalle': '',
   'precode': '        if not isinstance(nights, int) or nights < 1 or nights > 30:',
   'aftercode': '        if not isinstance(nights, int) or nights <= 1 or nights > 30:',
   'justificacion': ''},
  {'id': 4,
   'line': 23,
   'estado': 'killed',
   'aju

## 3. Marcar mutantes equivalentes

Un mutante **equivalente** cambia el código pero no su comportamiento observable:
ninguna entrada posible produce una salida distinta. Por definición no puede ser
matado por ningún test, así que dejarlo en el denominador castiga injustamente a
la batería.

Para cada sobreviviente, decide si es equivalente o si es una falla real de la
batería. Si sostienes que es equivalente, agrega dos campos a ese mutante en el
JSON y **vuelve a ejecutar** la celda de evaluación:

```json
{"id": 7, "line": 88, "precode": "...", "aftercode": "...",
 "equivalente": true,
 "justificacion": "0.0 y 0 son el mismo valor; rate solo se usa en una multiplicacion"}
```

Marcar un mutante como equivalente sin justificación no cuenta. Y ojo: marcar
como equivalente algo que en realidad revela un hueco en la batería es el error
más común de esta actividad.

**Si no encuentras ninguno, no inventes.** Que no haya equivalentes es un resultado
perfectamente normal, y no te resta puntaje: basta con decirlo en el reporte y
argumentar por qué cada sobreviviente es un hueco real de la batería. Marcar como
equivalente algo que sí es un hueco se penaliza igual que no analizarlo.


## 4. Baseline: mutantes clásicos

`generar_clasicos()` produce mutantes con operadores estándar de mutation testing
(ROR, AOR, COR, reemplazo de constantes) directamente sobre el código, usando
`tokenize`, así que nunca toca strings ni comentarios.

Este es el punto de comparación: los mutantes del LLM, ¿son más difíciles de
matar que los mecánicos? ¿O el modelo simplemente propone variaciones triviales
que la batería ya cubría?

In [9]:
clasicos = generar_clasicos()
print("mutantes clasicos generados:", len(clasicos))
evaluar("clasico", clasicos)

mutantes clasicos generados: 146
Mutantes de: CLASICO
----------------------------------------------------------------
  Total propuestos ................. 146
  Invalidos (no aplican) ........... 0
  Validos .......................... 146
  Equivalentes (marcados) .......... 0
  Killed ........................... 106
  Sobreviven ....................... 40
  Mutation score ajustado .......... 72.6%  (106 / 146)

  SOBREVIVIENTES (40) - analizalos en el reporte:
    #3   linea 21  
         antes : if not isinstance(nights, int) or nights < 1 or nights > 30:
         despues: if not isinstance(nights, int) or nights <= 1 or nights > 30:
    #6   linea 21  
         antes : if not isinstance(nights, int) or nights < 1 or nights > 30:
         despues: if not isinstance(nights, int) or nights < 2.0 or nights > 30:
    #8   linea 21  
         antes : if not isinstance(nights, int) or nights < 1 or nights > 30:
         despues: if not isinstance(nights, int) or nights < 1 or nights >= 30

{'nombre': 'clasico',
 'total': 146,
 'validos': 146,
 'invalidos': 0,
 'equivalentes': 0,
 'killed': 106,
 'sobreviven': 40,
 'denominador': 146,
 'score': 72.6,
 'filas': [{'id': 1,
   'line': 17,
   'estado': 'killed',
   'ajuste': None,
   'detalle': 'test_codigo_duplicado',
   'precode': '        if code in self.reservations:',
   'aftercode': '        if code not in self.reservations:',
   'justificacion': ''},
  {'id': 2,
   'line': 21,
   'estado': 'killed',
   'ajuste': None,
   'detalle': 'test_noches_bajo_minimo',
   'precode': '        if not isinstance(nights, int) or nights < 1 or nights > 30:',
   'aftercode': '        if not isinstance(nights, int) and nights < 1 or nights > 30:',
   'justificacion': ''},
  {'id': 3,
   'line': 21,
   'estado': 'sobrevive',
   'ajuste': None,
   'detalle': '',
   'precode': '        if not isinstance(nights, int) or nights < 1 or nights > 30:',
   'aftercode': '        if not isinstance(nights, int) or nights <= 1 or nights > 30:',
   '

## 5. Consolidación

**CELDA SELLADA — NO MODIFICAR.**

Imprime la tabla comparativa en Markdown, lista para pegar en `REPORTE.md`.

In [10]:
# ============================================================
# CELDA SELLADA - NO MODIFICAR
# ============================================================
consolidar()

## Tabla 1 - Mutantes por fuente

| Fuente | Total | Invalidos | Validos | Equivalentes | Killed | Sobreviven | Score ajustado |
|---|---|---|---|---|---|---|---|
| gpt | 20 | 0 | 20 | 0 | 10 | 10 | 50.0% |
| gemini | 20 | 0 | 20 | 0 | 10 | 10 | 50.0% |
| clasico | 146 | 0 | 146 | 0 | 106 | 40 | 72.6% |

----------------------------------------------------------------------
Sin observaciones automaticas.
Copia la tabla a REPORTE.md.
